# 2. The motif vocabulary: extraction, de-replication, and the vocabularies of 116 biosamples (Fig. 1e,f)

Three parts, all on **K562 (BSS00762)**:
1. **extract the motifs** (PWM + contextual weight of every first-layer filter) of the model trained in notebook 01 (GPU 10 min, or CPU 1 h);
2. **merge the motifs of the ten K562 models** into the non-redundant K562 motif set, from the shipped per-model motifs and their TOMTOM tables (`data/k562_demo/dereplication/`, CPU 15 min, no TOMTOM run needed);
3. **motif yield and catalogue concordance** of all 116 biosamples from the released motif sets (Zenodo `fig1_motif_sets_all116.tar.gz`, 100 MB, unpacked under `$ENHANCER_ROOT`): **Fig. 1e and 1f**.

Parts 1 and 2 need only the repository (part 1 falls back to the released K562 fold-0 model if notebook 01 was not run).

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"            # K562. Any of the 116 biosamples works once its two input files are in INPUTS (Zenodo fig1_training_inputs_all116)
N_MODELS = 1                  # models to train in section 1 (one per training fold); the paper trained 10
EPOCHS = 100                  # maximum epochs per model (paper: 100, early stopping with patience 10 on the validation fold)
SEED = 42
RUN_TRAINING = True           # False: skip training (no GPU) and use the released K562 fold-0 model and predictions in data/k562_sample
VALIDATE_EXTRACTION = True    # notebook 02, 1.3: extract from the released K562 fold-0 weights and compare with the released PWMs (10 min on a GPU)
RUN_TOMTOM_DEMO = True        # notebook 02, 3.3: run TOMTOM for the K562 promoter set against JASPAR here (about 4 min) and compare with the shipped table
INPUTS     = f"{ENHANCER_ROOT}/fig1_training_inputs" if BSSID != "BSS00762" else os.path.join(REPO, "data", "k562_demo")   # K562 ships in the repository; the other 115 biosamples: Zenodo fig1_training_inputs_all116 unpacked under ENHANCER_ROOT
MOTIF_SETS = f"{ENHANCER_ROOT}/fig1_motif_sets"         # Zenodo fig1_motif_sets_all116.tar.gz unpacked under ENHANCER_ROOT (section 4)
DEMO   = os.path.join(REPO, "data", "k562_demo")        # shipped: motifs + TOMTOM tables of the ten K562 models, K562 count matrices, K562 expression
SAMPLE = os.path.join(REPO, "data", "k562_sample")      # shipped: released K562 fold-0 model and the held-out predictions of the ten released models
WORK   = f"{ENHANCER_ROOT}/notebook_runs/fig1/{BSSID}"; os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, f"{CODE}/cnncr_model"); sys.path.insert(0, f"{CODE}/motif_extraction")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, logomaker
from scipy.stats import spearmanr, pearsonr
from IPython.display import Image, display
from cnncr_io import read_windows_fasta, read_expression, one_hot      # code/cnncr_model/cnncr_io.py
import cnncr_model as cm                                                # code/cnncr_model/cnncr_model.py: the model, split and training loop of the paper
from extract_motifs_lib import extract_model, write_meme, read_meme     # code/motif_extraction/extract_motifs_lib.py
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device, "| work folder:", WORK)
PRO, PRO_F, ENH, ENH_F = "#8FB8C9", "#D6E7EE", "#D79581", "#F4E0D8"        # promoter / enhancer colours of the paper's figures
def show_pwms(pwms, titles, suptitle=""):
    """sequence logos (information content) of a few PWMs"""
    fig, axes = plt.subplots(1, len(pwms), figsize=(2.0*len(pwms), 1.3))
    for ax, p, t in zip(np.atleast_1d(axes), pwms, titles):
        logomaker.Logo(logomaker.transform_matrix(pd.DataFrame(p, columns=list("ACGT")), from_type="probability", to_type="information"), ax=ax)
        ax.set_title(t, fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(suptitle, fontsize=9); plt.tight_layout(); plt.show()
def spearman(a, b): return spearmanr(a, b)[0]
def same_pwms(a, b):
    """the same PWMs regardless of order (each PWM rounded to 6 decimals)"""
    key = lambda m: np.round(m, 6).tobytes(); ka, kb = set(map(key, a)), set(map(key, b))
    return f"{len(ka & kb)} of {len(kb)} released PWMs recovered" + (" (identical set)" if len(a) == len(b) and ka == kb else "")

## 1. Extract the motifs learned by the model

Every first-layer filter is turned into a motif in three steps (`extract_motifs_lib.py`, the procedure of the paper's `extract_CNN_CR_motifs_pytorch.py`):
1. run all genes' windows of a branch through its first convolution (activations of the 256 filters at 393 positions) and through its contextual-weight module;
2. **PWM**: for each filter, the 8-mers under its 1,000 strongest |activations| over all genes, as position frequencies;
3. **weight**: the median of (contextual weight × mean activation) over the genes where that product exceeds half its maximum.

Branches 0-9 give the 2,560 promoter PWMs of the model, branches 10-19 the 2,560 enhancer PWMs. The extraction uses **all** genes in their original order (this is a description of the model, not a prediction). About 10 min on a GPU, an hour on a CPU; the activations of one branch take 7 GB of RAM.

### 1.1 Run it on the model trained in notebook 01 (fold 0)

In [ ]:
pth = f"{WORK}/{BSSID}_it_0_256MOTIF_pytorch.pth"                      # the fold-0 model trained in notebook 01 ...
if not os.path.exists(pth): pth = f"{SAMPLE}/BSS00762_it_0_256MOTIF_pytorch.pth"; print("no model from notebook 01 in the work folder: using the released K562 fold-0 model")
model = cm.MultiBranchModel(num_branches=20); model.load_state_dict(torch.load(pth, map_location="cpu")); model.to(device); print("model:", pth)
X, _ = read_windows_fasta(f"{INPUTS}/{BSSID}.windows.fa.gz", n_genes=18356)      # all genes, original order (the input of notebook 01)
t0 = time.time(); motifs = extract_model(model, X, device); print(f"extraction: {(time.time()-t0)/60:.1f} min")
for t in ("pro", "enh"):
    pwms, w = motifs[t]
    write_meme(f"{WORK}/{BSSID}_it_0_256MOTIF_pytorch_0_{t}_motifs_all_256.meme", pwms); np.save(f"{WORK}/{BSSID}_it_0_256MOTIF_pytorch_0_{t}_motifs_weights_all_256.npy", w)
    print(f"{t}: {len(pwms)} PWMs, weights: {np.sum(w > 0)} positive, max {w.max():.3g}")

### 1.2 What the filters learned
A few filters carry most of the weight; the highest-weight promoter filters of a K562 model are typically the GC-box / SP-KLF, NRF1, ETS and CCAAT-box families.

In [ ]:
pwms, w = motifs["pro"]
fig, ax = plt.subplots(figsize=(4, 2)); ax.plot(np.sort(w)[::-1], color=PRO, lw=1.2); ax.set_xlabel("promoter filter, ranked"); ax.set_ylabel("weight"); plt.tight_layout(); plt.show()
top = np.argsort(-w)[:6]; show_pwms([pwms[i] for i in top], [f"filter {i}\nw = {w[i]:.2g}" for i in top], "six highest-weight promoter filters of the fold-0 model")
pwms_e, w_e = motifs["enh"]; top = np.argsort(-w_e)[:6]; show_pwms([pwms_e[i] for i in top], [f"filter {i}\nw = {w_e[i]:.2g}" for i in top], "six highest-weight enhancer filters")

### 1.3 Check against the released extraction
The released per-model motif files of K562 are in `data/k562_demo/dereplication/`. Extracting from the released fold-0 weights (`data/k562_sample/`) must give the same PWMs: promoter PWMs identical and in the same order; the released enhancer PWMs of each branch were stored sorted by weight (an older run of the script), so they are compared as sorted sets. Ties among the 1,000 strongest activations (frequent in enhancer branches, where half of the windows are absent and their activations are exactly zero) are broken differently by different GPUs, so a few percent of the enhancer PWMs differ slightly from the released ones; the weights are unaffected.

In [ ]:
if VALIDATE_EXTRACTION:
    m0 = cm.MultiBranchModel(num_branches=20); m0.load_state_dict(torch.load(f"{SAMPLE}/BSS00762_it_0_256MOTIF_pytorch.pth", map_location="cpu")); m0.to(device)
    t0 = time.time(); ref = extract_model(m0, X, device, verbose=False); print(f"extraction from the released fold-0 weights: {(time.time()-t0)/60:.1f} min")
    for t in ("pro", "enh"):
        pw, w = ref[t]; rel = read_meme(f"{DEMO}/dereplication/BSS00762_it_0_256MOTIF_pytorch_0_{t}_motifs_all_256.meme"); wrel = np.load(f"{DEMO}/dereplication/BSS00762_it_0_256MOTIF_pytorch_0_{t}_motifs_weights_all_256.npy")
        if t == "enh":     # released enhancer PWMs: sorted by weight within each branch of 256
            order = np.concatenate([b * 256 + np.argsort(-w[b*256:(b+1)*256]) for b in range(len(pw) // 256)]); pw = [pw[i] for i in order]; w = w[order]
        same = sum(np.abs(a - b).max() < 1e-6 for a, b in zip(pw, rel)); close = sum(np.abs(a - b).max() < 0.1 for a, b in zip(pw, rel))
        print(f"{t}: {len(pw)} extracted vs {len(rel)} released | identical PWMs {same}, within 0.1 {close} | weights: max |diff| {np.abs(w - wrel[:len(w)]).max():.2g}, Spearman {spearman(w, wrel[:len(w)]):.4f}")

## 2. From ten models to one non-redundant motif set (Fig. 1b, step 3)

Each of the ten models of a biosample yields 2,560 promoter and 2,560 enhancer PWMs, most of them variants of the same motif. The paper's de-replication (`code/motif_extraction/combine_motifs_1.py`, `combine_motifs_2.py`) has four steps:
1. **TOMTOM of every model's PWMs against themselves** (`tomtom -no-ssc -text -dist pearson -evalue -thresh 5 model.meme model.meme`);
2. **within each model** (`combine_motifs_1.py BSSID pro|enh FOLDER`): keep the 750 highest-weight filters with information content > 10 bits, connect filters that TOMTOM calls similar (E ≤ 1, p ≤ 0.05, offset 0), keep one representative per connected group, and pool the ten models → `{BSSID}_{type}_motifs_combined.meme` (+ the weights of every member);
3. **TOMTOM of the pooled set against itself**;
4. **across models** (`combine_motifs_2.py BSSID pro|enh FOLDER`): group the pooled motifs by the same similarity rule and keep a group when it was found by at least two models → `{BSSID}_{type}_motifs_nonredundant.meme` and the median weight of each motif.

The TOMTOM steps (1 and 3) take about 10 CPU-hours per biosample, so `data/k562_demo/dereplication/` ships the ten K562 models' PWMs and weights (the output of part 1 of this notebook, for every model) **together with their TOMTOM tables**; here steps 2 and 4 are run on them and the result is compared with the released K562 motif set. To de-replicate your own models, run the TOMTOM commands printed at the end of this section first.

In [ ]:
DR = f"{WORK}/dereplication"; os.makedirs(DR, exist_ok=True)
for f in glob.glob(f"{DEMO}/dereplication/*"):
    if os.path.isfile(f): shutil.copy(f, DR)
print(open(f"{DEMO}/dereplication/README.txt").read())
print("per-model PWM files:", len(glob.glob(f"{DR}/*_motifs_all_256.meme")), "| per-model TOMTOM tables:", len(glob.glob(f"{DR}/*_motifs_all_tomtom.txt.gz")))

### 2.1 Step 2: within-model grouping and pooling of the ten models (about 5 min per window type)

In [ ]:
ME = f"{CODE}/motif_extraction"
for t in ("pro", "enh"): run(f"python combine_motifs_1.py {BSSID} {t} {DR}", cwd=ME)
for t in ("pro", "enh"):
    new = read_meme(f"{DR}/{BSSID}_{t}_motifs_combined.meme"); rel = read_meme(f"{DEMO}/dereplication/released/{BSSID}_{t}_motifs_combined.meme")
    print(f"{t}: pooled representatives {len(new)} (released {len(rel)}); {same_pwms(new, rel)}")

### 2.2 Step 4: cross-model grouping → the non-redundant K562 motif set
Step 3 (TOMTOM of the pooled set) is read from the shipped table `{BSSID}_{type}_motifs_combined_1_tomtom.txt.gz`, which was computed on the pooled set that step 2 produces here.

The regenerated K562 **promoter** set is not identical to the released one (729 motifs against 736 released, with 567 of the released PWMs recovered); the **enhancer** set is reproduced exactly. The cause is in the shipped inputs, not in the code run here. The per-model TOMTOM tables in `data/k562_demo/dereplication/` were computed on a different extraction run of the same ten models than the per-model PWM files shipped beside them: 98 of the 25,600 promoter PWMs and 71 of the 25,600 enhancer PWMs have a different consensus sequence in the TOMTOM table than in the PWM file, because the extraction breaks ties among the 1,000 strongest activations differently on different hardware. `combine_motifs_1.py` and `combine_motifs_2.py` are fully deterministic and no GPU or TOMTOM run is involved at this point, but the similarity graph is built from a table that does not exactly describe the PWMs it is applied to, so a few groups gain or lose a member and the count of representatives shifts.

Nothing downstream depends on this: the released sets are the ones used everywhere in the paper (Zenodo `fig1_motif_sets_all116`), Fig. 1e,f are computed from them, and notebook 04 regenerates the 43 core motifs from them as an identical set.

In [ ]:
for t in ("pro", "enh"): run(f"python combine_motifs_2.py {BSSID} {t} {DR}", cwd=ME)
for t in ("pro", "enh"):
    new = read_meme(f"{DR}/{BSSID}_{t}_motifs_nonredundant.meme"); rel = read_meme(f"{DEMO}/dereplication/released/{BSSID}_{t}_motifs_nonredundant.meme")
    print(f"{t}: non-redundant motifs {len(new)} (released {len(rel)}); {same_pwms(new, rel)}")
w = np.load(f"{DR}/{BSSID}_pro_motifs_nonredundant_weights.npy"); pw = read_meme(f"{DR}/{BSSID}_pro_motifs_nonredundant.meme"); top = np.argsort(-w)[:6]
show_pwms([pw[i] for i in top], [f"motif {i}\nw = {w[i]:.2g}" for i in top], "six highest-weight non-redundant K562 promoter motifs")

### 2.3 For your own models: the TOMTOM commands
With the ten models trained (notebook 01, `N_MODELS = 10`) and their motifs extracted (part 1 of this notebook, for every model), the per-model TOMTOM tables of step 1 and, after step 2, the pooled-set table of step 3 are produced by the commands below (written to `tomtom_commands.sh` in the work folder; for all 116 biosamples they were run as one cluster job per biosample).

In [ ]:
TT = "tomtom -no-ssc -verbosity 2 -text -dist pearson -evalue -thresh 5"; cmds = []
for i in range(10):
    for t in ("pro", "enh"):
        m = f"{DR}/{BSSID}_it_{i}_256MOTIF_pytorch_{i}_{t}_motifs_all_256.meme"; cmds.append(f"{TT} {m} {m} > {DR}/{BSSID}_it_{i}_256MOTIF_pytorch_{t}_motifs_all_tomtom.txt")
cmds.append(f"# after combine_motifs_1.py:"); [cmds.append(f"{TT} {DR}/{BSSID}_{t}_motifs_combined.meme {DR}/{BSSID}_{t}_motifs_combined.meme > {DR}/{BSSID}_{t}_motifs_combined_1_tomtom.txt") for t in ("pro", "enh")]
open(f"{DR}/tomtom_commands.sh", "w").write("\n".join(cmds) + "\n"); print(*cmds[:2], "...", *cmds[-3:], sep="\n")

## 3. The vocabularies of 116 biosamples: Fig. 1e and 1f

Part 2 was run for every biosample; the 116 resulting motif sets are Zenodo `fig1_motif_sets_all116.tar.gz` (`$ENHANCER_ROOT/fig1_motif_sets/`), together with the TOMTOM comparison of every set against four motif catalogues: **JASPAR 2022** (CORE vertebrates), **CIS-BP 2.0** (human) and the two **Epimotifs** catalogues (DNAmeth, Epigram), all four shipped in `data/motifs/`.

### 3.1 Fig. 1e: motifs per biosample

In [ ]:
bios = pd.read_csv(f"{MOTIF_SETS}/biosamples.tsv", sep="\t"); print(len(bios), "biosamples; first:", ", ".join(bios.name[:4]))
counts = pd.DataFrame({t: [n_motifs(f"{MOTIF_SETS}/{b}_{t}_motifs_nonredundant.meme") for b in bios.BSSID] for t in ("pro", "enh")}, index=bios.BSSID)
print("median motifs per biosample: promoter %d, enhancer %d (paper: 705 and 684)" % (counts.pro.median(), counts.enh.median()))
fig, ax = plt.subplots(figsize=(2.6, 2.8)); rng = np.random.RandomState(0)
for k, (t, c, cf, lab) in enumerate((("pro", PRO, PRO_F, "promoter"), ("enh", ENH, ENH_F, "enhancer"))):
    ax.scatter(k + rng.uniform(-0.22, 0.22, len(counts)), counts[t], s=6, color=c, alpha=0.6, lw=0, zorder=1)
    bp = ax.boxplot([counts[t]], positions=[k], widths=0.55, patch_artist=True, showfliers=False, zorder=2)
    bp["boxes"][0].set(facecolor=cf, edgecolor=c, alpha=0.85); [l.set(color=c) for l in bp["whiskers"] + bp["caps"] + bp["medians"]]
    ax.text(k, counts[t].median(), f"{counts[t].median():.0f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(["promoter", "enhancer"]); ax.set_ylabel("non-redundant motifs per biosample"); ax.set_title(f"n = {len(counts)} biosamples", fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); fig.savefig(f"{WORK}/fig1e.png", dpi=200, transparent=True); plt.show()

### 3.2 Fig. 1f: concordance with the motif catalogues
A de novo motif is *catalogued* when TOMTOM matches it to a catalogue motif with **E ≤ 1 and p ≤ 0.01**. Per biosample: the fraction of its motifs catalogued in JASPAR or CIS-BP, and the fraction catalogued in any of the four catalogues (JASPAR, CIS-BP, Epimotifs DNAmeth, Epimotifs Epigram). The faint lines join the two values of one biosample.

In [ ]:
DBS = ("JASPAR", "CISBP", "DNAmeth", "Epigram")
def catalogued(path):
    """TOMTOM table -> the de novo motif ids (query) with a match at E <= 1 and p <= 0.01"""
    d = pd.read_csv(path, sep="\t", comment="#", usecols=["Query_ID", "E-value", "p-value"]); d = d[(d["E-value"] <= 1) & (d["p-value"] <= 0.01)]
    return set(d["Query_ID"].astype(str))
frac = {}
for t in ("pro", "enh"):
    rows = []
    for b in bios.BSSID:
        ids = {db: catalogued(f"{MOTIF_SETS}/catalogue_tomtom/{db}/{b}_{t}_tomtom_{db}.txt.gz") for db in DBS}; n = counts.loc[b, t]
        rows.append((len(ids["JASPAR"] | ids["CISBP"]) / n, len(set.union(*ids.values())) / n))
    frac[t] = np.array(rows)
    print(f"{t}: median fraction catalogued in JASPAR/CIS-BP {np.median(frac[t][:, 0]):.3f}, in all four catalogues {np.median(frac[t][:, 1]):.3f}")
print("paper: promoter 62% -> 94%, enhancer 71% -> 95%")
fig, ax = plt.subplots(figsize=(3.4, 2.8))
for k, (t, c, cf, lab) in enumerate((("pro", PRO, PRO_F, "promoter"), ("enh", ENH, ENH_F, "enhancer"))):
    x0, x1 = 2.6*k, 2.6*k + 1; f = frac[t] * 100
    for a, b2 in f: ax.plot([x0, x1], [a, b2], color="#66737c", lw=0.4, alpha=0.35, zorder=1)
    for x, v, fc in ((x0, f[:, 0], "white"), (x1, f[:, 1], cf)):
        bp = ax.boxplot([v], positions=[x], widths=0.6, patch_artist=True, showfliers=False, zorder=3)
        bp["boxes"][0].set(facecolor=fc, edgecolor=c); [l.set(color=c) for l in bp["whiskers"] + bp["caps"] + bp["medians"]]
        ax.text(x, np.median(v) + 1, f"{np.median(v):.0f}%", ha="center", va="bottom", fontsize=7)
ax.set_xticks([0, 1, 2.6, 3.6]); ax.set_xticklabels(["JASPAR\n+CIS-BP", "+ Epi-\nmotifs"] * 2, fontsize=7); ax.set_ylabel("% de novo motifs catalogued")
ax.text(0.5, 101, "promoter", ha="center", fontsize=8, color=PRO); ax.text(3.1, 101, "enhancer", ha="center", fontsize=8, color=ENH); ax.set_ylim(30, 105)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); fig.savefig(f"{WORK}/fig1f.png", dpi=200, transparent=True); plt.show()

### 3.3 Running TOMTOM yourself
The shipped tables were made with `tomtom -no-ssc -verbosity 2 -text -dist pearson -evalue -thresh 5 <motif set> <catalogue>` against the four catalogues in `data/motifs/` (`JASPAR2022_CORE_vertebrates_non_redundant_unique_names.meme`, `CISBP_human_motifs.meme`, `Epimotifs_DNAmeth_human.meme`, `Epimotifs_Epigram_human.meme`); one comparison of a ~700-motif set with JASPAR takes about 4 min, so the 116 × 2 × 4 tables are about 60 CPU-hours (a cluster job over biosamples). Here the K562 promoter set is compared with JASPAR and the fraction catalogued is checked against the shipped table. If part 2 was run, the K562 set regenerated there is the same file.

In [ ]:
if RUN_TOMTOM_DEMO and shutil.which("tomtom"):
    ms, out = f"{MOTIF_SETS}/{BSSID}_pro_motifs_nonredundant.meme", f"{WORK}/{BSSID}_pro_tomtom_JASPAR.txt"
    run(f"tomtom -no-ssc -verbosity 1 -text -dist pearson -evalue -thresh 5 {ms} {REPO}/data/motifs/JASPAR2022_CORE_vertebrates_non_redundant_unique_names.meme > {out}", cwd=WORK)
    n = n_motifs(ms); print(f"{BSSID} promoter motifs catalogued in JASPAR: here {len(catalogued(out))/n:.3f}, shipped table {len(catalogued(f'{MOTIF_SETS}/catalogue_tomtom/JASPAR/{BSSID}_pro_tomtom_JASPAR.txt.gz'))/n:.3f}")
else: print("skipped (RUN_TOMTOM_DEMO = False or tomtom not on PATH)")